# Model Evaluation and Tuning

Real-data evaluation notebook for the battery surrogate models.

This version uses the existing OP bundle pipeline, real train/val/test OP subsets, fixed step spacing of `TIME_DELTA_S = 0.1 s`, and a single shared progress counter across the benchmark suite.

The benchmark subset is intentionally small so the notebook remains runnable on CPU while still exercising the real collectors, trainers, and sequence evaluators.

In [17]:
from __future__ import annotations

import copy
import importlib
import json
import sys
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

repo_root = Path.cwd().resolve()
while not (repo_root / "src").exists():
    if repo_root.parent == repo_root:
        raise RuntimeError("Could not locate the repository root from the notebook working directory.")
    repo_root = repo_root.parent

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

import battery_surrogate.model.evaluate_sequence as evaluate_sequence_module
importlib.reload(evaluate_sequence_module)
import battery_surrogate.model.recurrent_pointwise as recurrent_pointwise_module
importlib.reload(recurrent_pointwise_module)
import battery_surrogate.model.registry as registry_module
importlib.reload(registry_module)
import battery_surrogate.cli.train as train_module
importlib.reload(train_module)


from battery_surrogate.cli.train import train_from_config
from battery_surrogate.data.loader import load_op
from battery_surrogate.model.evaluate import collect_pointwise_predictions
from battery_surrogate.model.evaluate import _metric_bundle as metric_bundle_pointwise
from battery_surrogate.model.evaluate_sequence import benchmark_history_lengths, collect_sequence_predictions
from battery_surrogate.model.evaluate_sequence import _metric_bundle as metric_bundle_sequence
from battery_surrogate.model.features_pointwise import iter_pointwise_blocks
from battery_surrogate.model.features_sequence import resolve_history_lengths
from battery_surrogate.model.normalizer import PointwiseNormalizer
from battery_surrogate.model.progress import make_progress_printer
from battery_surrogate.model.registry import build_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
TIME_DELTA_S = 0.1  # Assumed step spacing: length * TIME_DELTA_S = seconds.
TS_EXTRAPOLATION = "clamp"

BENCH_TRAIN_OPS = ["OP01", "OP02", "OP08"]
BENCH_VAL_OPS = ["OP09", "OP10"]
BENCH_TEST_OPS = ["OP11", "OP12"]
BENCH_OPS = {
    "train": BENCH_TRAIN_OPS,
    "val": BENCH_VAL_OPS,
    "test": BENCH_TEST_OPS,
}

BASE_MLP_CONFIG = yaml.safe_load((repo_root / "configs" / "model" / "mlp_pointwise.yaml").read_text(encoding="utf-8"))
BASE_RECURRENT_CONFIG = yaml.safe_load((repo_root / "configs" / "model" / "recurrent_pointwise.yaml").read_text(encoding="utf-8"))

ARTIFACT_ROOT = repo_root / "artifacts" / "real_data_eval_benchmarks"
SHARED_NORMALIZER_PATH = ARTIFACT_ROOT / "shared_normalizer.json"

SCATTER_MLP_EPOCHS = 8
SCATTER_RECURRENT_EPOCHS = 8
CAPACITY_EPOCHS = 4
BETA_EPOCHS = 4
HISTORY_EPOCHS = 3
EPOCH_PLOT_EPOCHS = 50
HIST_STEP = 1

CAPACITY_WIDTHS = [32, 64, 128, 256, 512]
CAPACITY_DEPTHS = [1, 2, 3, 4, 5]
BETA_VALUES = [0.5, 1.0, 1.5, 2.0]
HISTORY_K_VALUES = list(range(1, 51, HIST_STEP))

TOTAL_RUNS = (
    2
    + len(CAPACITY_WIDTHS)
    + len(CAPACITY_DEPTHS)
    + len(BETA_VALUES)
    + len(HISTORY_K_VALUES)
    + 1
)

# ============================================================================
# MODEL CONFIGURATION LABEL HELPER
# ============================================================================
# Standard model configuration for labeling all plots

def make_model_label(
    model_type: str = "MLP pointwise",
    hidden_layers: int = 3,
    hidden_size: int = 128,
    activation: str = "LearnableSwish (beta=1.0 fixed)",
    epochs: int = 8,
    batch_size: int = 4096,
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    grad_clip: float = 1.0,
    early_stopping: int = 10,
    train_ops: list[str] | None = None,
    val_ops: list[str] | None = None,
    test_ops: list[str] | None = None,
    T_weight: float = 1.0,
    bc_V_weight: float = 1.0,
    multiline: bool = True,
) -> str:
    """Generate standardized model configuration label for plots."""
    if train_ops is None:
        train_ops = BENCH_TRAIN_OPS
    if val_ops is None:
        val_ops = BENCH_VAL_OPS
    if test_ops is None:
        test_ops = BENCH_TEST_OPS

    if multiline:
        return (
            f"Model: {model_type} | Layers: {hidden_layers} | Width: {hidden_size} | Act: {activation}\n"
            f"Epochs: {epochs} | Batch: {batch_size} | LR: {lr} | WD: {weight_decay} | Clip: {grad_clip} | ES: {early_stopping}\n"
            f"Train: {', '.join(train_ops)} | Val: {', '.join(val_ops)} | Test: {', '.join(test_ops)}\n"
            f"Loss: Weighted MSE (T_weight={T_weight}, bc_V_weight={bc_V_weight})"
        )
    else:
        return (
            f"Model: {model_type} | {hidden_layers}×{hidden_size} | {activation} | "
            f"Train: {', '.join(train_ops)} | Test: {', '.join(test_ops)}"
        )


def make_recurrent_label(
    hidden_layers: int = 2,
    hidden_size: int = 128,
    rnn_type: str = "GRU",
    k_T: int = 8,
    k_V: int = 4,
    epochs: int = 8,
    batch_size: int = 1,
    train_ops: list[str] | None = None,
    val_ops: list[str] | None = None,
    test_ops: list[str] | None = None,
    T_weight: float = 1.0,
    bc_V_weight: float = 1.0,
) -> str:
    """Generate standardized recurrent model configuration label for plots."""
    if train_ops is None:
        train_ops = BENCH_TRAIN_OPS
    if val_ops is None:
        val_ops = BENCH_VAL_OPS
    if test_ops is None:
        test_ops = BENCH_TEST_OPS

    return (
        f"Model: Recurrent ({rnn_type}) | Layers: {hidden_layers} | Width: {hidden_size} | History: k_T={k_T}, k_V={k_V}\n"
        f"Epochs: {epochs} | Batch: {batch_size} | Seed: {SEED}\n"
        f"Train: {', '.join(train_ops)} | Val: {', '.join(val_ops)} | Test: {', '.join(test_ops)}\n"
        f"Loss: Weighted MSE (T_weight={T_weight}, bc_V_weight={bc_V_weight})"
    )


# Default MLP label for quick reference
DEFAULT_MLP_LABEL = make_model_label(
    model_type="MLP pointwise (not recurrent)",
    hidden_layers=3,
    hidden_size=128,
    activation="LearnableSwish (beta=1.0 fixed)",
    epochs=8,
    batch_size=4096,
    lr=1e-3,
    weight_decay=0.0,
    grad_clip=1.0,
    early_stopping=10,
)

print("=" * 80)
print("DEFAULT MODEL CONFIGURATION")
print("=" * 80)
print(DEFAULT_MLP_LABEL)
print("=" * 80)

# ============================================================================

_progress_state = {"done": 0}
_progress_printer = make_progress_printer(TOTAL_RUNS, desc="Full benchmark suite", use_tqdm=True)


def advance_progress(steps: int = 1, msg: str = "") -> None:
    _progress_state["done"] += steps
    _progress_printer(_progress_state["done"], TOTAL_RUNS, msg)


def make_progress_proxy() -> callable:
    state = {"done": 0}

    def proxy(done: int, total: int | None = None, msg: str = "") -> None:
        delta = done - state["done"]
        if delta > 0:
            state["done"] = done
            advance_progress(delta, msg)

    return proxy


def load_base_config(path: Path) -> dict:
    payload = yaml.safe_load(path.read_text(encoding="utf-8"))
    return payload if isinstance(payload, dict) else {}


def build_benchmark_config(
    base_config: dict,
    *,
    model_type: str,
    train_ops: list[str],
    val_ops: list[str],
    test_ops: list[str],
    epochs: int,
    output_tag: str,
    model_overrides: dict | None = None,
) -> dict:
    config = copy.deepcopy(base_config)
    config.setdefault("seed", SEED)
    config.setdefault("data", {})
    config.setdefault("model", {})
    config.setdefault("train", {})
    config.setdefault("loss", {"T_weight": 1.0, "bc_V_weight": 1.0})
    config.setdefault("output", {})

    config["seed"] = SEED
    config["data"].update(
        {
            "train_ops": list(train_ops),
            "val_ops": list(val_ops),
            "test_ops": list(test_ops),
            "subsample_time": int(config["data"].get("subsample_time", 50)),
            "ts_extrapolation": TS_EXTRAPOLATION,
        }
    )
    config["model"]["type"] = model_type
    if model_overrides:
        config["model"].update(model_overrides)
    config["train"]["epochs"] = int(epochs)
    config["output"]["ckpt_dir"] = str(ARTIFACT_ROOT / output_tag / "{timestamp}")
    return config


def fit_shared_normalizer(train_ops: list[str], *, subsample_time: int, ts_extrapolation: str) -> PointwiseNormalizer:
    normalizer = PointwiseNormalizer()
    for op_id in train_ops:
        bundle = load_op(op_id)
        time_indices = np.arange(0, bundle.t_fast.shape[0], subsample_time, dtype=np.int64)
        for x_block, y_block, _, _ in iter_pointwise_blocks(
            bundle,
            time_indices,
            ts_extrapolation=ts_extrapolation,
        ):
            normalizer.partial_fit(x_block, y_block)
    normalizer.finalize()
    return normalizer


def latest_artifact_dir(output_tag: str) -> Path | None:
    root = ARTIFACT_ROOT / output_tag
    if not root.exists():
        return None
    candidates = [path for path in root.iterdir() if path.is_dir() and (path / "config.yaml").exists() and (path / "best.pt").exists()]
    if not candidates:
        return None
    return max(candidates, key=lambda path: path.stat().st_mtime)


def train_or_load_summary(config: dict, output_tag: str, progress_label: str) -> dict:
    latest_dir = latest_artifact_dir(output_tag)
    if latest_dir is not None:
        saved_config = yaml.safe_load((latest_dir / "config.yaml").read_text(encoding="utf-8"))
        best_state = torch.load(latest_dir / "best.pt", map_location="cpu")
        advance_progress(1, f"{progress_label} (cached)")
        return {
            "best_val_loss": float(best_state["best_val_loss"]),
            "ckpt_dir": str(latest_dir),
            "best_ckpt": str(latest_dir / "best.pt"),
            "normalizer": str(latest_dir / "normalizer.json"),
            "config_path": str(latest_dir / "config.yaml"),
            "n_parameters": 0,
            "model_type": str(saved_config.get("model", {}).get("type", "mlp_pointwise")),
            "n_sensors": int(saved_config.get("n_sensors", 363)),
        }
    summary = train_from_config(config)
    advance_progress(1, progress_label)
    return summary


def load_trained_artifacts(summary: dict) -> tuple[dict, torch.nn.Module, PointwiseNormalizer]:
    saved_config = yaml.safe_load(Path(summary["config_path"]).read_text(encoding="utf-8"))
    model = build_model(saved_config, n_sensors=int(summary["n_sensors"]), seed=SEED)
    checkpoint = torch.load(summary["best_ckpt"], map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    model = model.to(DEVICE)
    model.eval()
    normalizer = PointwiseNormalizer.load(Path(summary["normalizer"]))
    return saved_config, model, normalizer


if SHARED_NORMALIZER_PATH.exists():
    shared_normalizer = PointwiseNormalizer.load(SHARED_NORMALIZER_PATH)
else:
    shared_normalizer = fit_shared_normalizer(
        BENCH_TRAIN_OPS,
        subsample_time=int(BASE_MLP_CONFIG.get("data", {}).get("subsample_time", 50)),
        ts_extrapolation=TS_EXTRAPOLATION,
    )
    SHARED_NORMALIZER_PATH.parent.mkdir(parents=True, exist_ok=True)
    shared_normalizer.save(SHARED_NORMALIZER_PATH)

print(f"Repo root: {repo_root}")
print(f"Device: {DEVICE}")
print(f"Shared normalizer: {SHARED_NORMALIZER_PATH if SHARED_NORMALIZER_PATH.exists() else 'fit in memory'}")
print(f"TOTAL_RUNS = {TOTAL_RUNS}")
print(f"Benchmark split: train={BENCH_TRAIN_OPS}, val={BENCH_VAL_OPS}, test={BENCH_TEST_OPS}")

DEFAULT MODEL CONFIGURATION
Model: MLP pointwise (not recurrent) | Layers: 3 | Width: 128 | Act: LearnableSwish (beta=1.0 fixed)
Epochs: 8 | Batch: 4096 | LR: 0.001 | WD: 0.0 | Clip: 1.0 | ES: 10
Train: OP01, OP02, OP08 | Val: OP09, OP10 | Test: OP11, OP12
Loss: Weighted MSE (T_weight=1.0, bc_V_weight=1.0)


Test 2 capacity study: 60it [41:58, 41.98s/it, Section3B depth=5, beta=2.0]

Repo root: C:\Users\M0245635\batterysurrogatemodell\battery_surrogate_agenticWorkflow
Device: cpu
Shared normalizer: C:\Users\M0245635\batterysurrogatemodell\battery_surrogate_agenticWorkflow\artifacts\real_data_eval_benchmarks\shared_normalizer.json
TOTAL_RUNS = 67
Benchmark split: train=['OP01', 'OP02', 'OP08'], val=['OP09', 'OP10'], test=['OP11', 'OP12']


## Test 2 - Capacity Study (MLP only)

This section isolates architecture capacity for the pointwise MLP on real OP data.

What changes in this test:
- Width sweep: `hidden_size in CAPACITY_WIDTHS` with fixed depth `n_hidden_layers = 3`
- Depth sweep: `n_hidden_layers in CAPACITY_DEPTHS` with fixed width `hidden_size = 128`

What stays fixed:
- Split: train = OP01, OP02, OP08; val = OP09, OP10; test = OP11, OP12
- Swish beta fixed at `beta = 1.0` and `swish_beta_learnable = False`
- Epoch budget per run: `CAPACITY_EPOCHS`
- Same seed, same loss weights, same CPU-runnable settings

Outputs:
- Validation loss vs capacity
- Parameter count tables for width and depth sweeps

In [ ]:
# -----------------------------
# Test 2: MLP capacity study
# -----------------------------

TEST2_TOTAL_RUNS = len(CAPACITY_WIDTHS) + len(CAPACITY_DEPTHS)
TOTAL_RUNS = TEST2_TOTAL_RUNS
_progress_state = {"done": 0}
_progress_printer = make_progress_printer(TOTAL_RUNS, desc="Test 2 capacity study", use_tqdm=True)


def make_plot_annotation(
    model_type: str,
    arch_dict: dict,
    train_dict: dict,
    data_split_dict: dict,
    loss_dict: dict,
) -> str:
    """Build a consistent annotation text block for benchmark plots."""
    arch_text = ", ".join(f"{k}={v}" for k, v in arch_dict.items())
    train_text = ", ".join(f"{k}={v}" for k, v in train_dict.items())
    data_text = (
        f"train={data_split_dict['train']} | val={data_split_dict['val']} | test={data_split_dict['test']}"
    )
    loss_text = ", ".join(f"{k}={v}" for k, v in loss_dict.items())
    return (
        f"Model: {model_type}\n"
        f"Arch: {arch_text}\n"
        f"Train: {train_text}\n"
        f"Data: {data_text}\n"
        f"Loss: weighted MSE ({loss_text})"
    )


def add_annotation_box(ax, text: str) -> None:
    ax.text(
        0.98,
        0.02,
        text,
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=7,
        bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "gray"},
    )


width_rows: list[dict] = []
for width in CAPACITY_WIDTHS:
    cfg = build_benchmark_config(
        BASE_MLP_CONFIG,
        model_type="mlp_pointwise",
        train_ops=BENCH_TRAIN_OPS,
        val_ops=BENCH_VAL_OPS,
        test_ops=BENCH_TEST_OPS,
        epochs=CAPACITY_EPOCHS,
        output_tag=f"test2/capacity/width_{width}",
        model_overrides={
            "n_hidden_layers": 3,
            "hidden_size": int(width),
            "swish_beta_init": 1.0,
            "swish_beta_learnable": False,
        },
    )

    train_start = perf_counter()
    summary = train_or_load_summary(cfg, f"test2/capacity/width_{width}", f"Test2 width={width}")
    train_s = perf_counter() - train_start

    saved_config, model, _normalizer = load_trained_artifacts(summary)
    params = int(sum(p.numel() for p in model.parameters()))
    width_rows.append(
        {
            "width": int(width),
            "best_val_loss": float(summary["best_val_loss"]),
            "train_s": float(train_s),
            "params": int(params),
            "layers": int(saved_config.get("model", {}).get("n_hidden_layers", 3)),
        }
    )


depth_rows: list[dict] = []
for depth in CAPACITY_DEPTHS:
    cfg = build_benchmark_config(
        BASE_MLP_CONFIG,
        model_type="mlp_pointwise",
        train_ops=BENCH_TRAIN_OPS,
        val_ops=BENCH_VAL_OPS,
        test_ops=BENCH_TEST_OPS,
        epochs=CAPACITY_EPOCHS,
        output_tag=f"test2/capacity/depth_{depth}",
        model_overrides={
            "n_hidden_layers": int(depth),
            "hidden_size": 128,
            "swish_beta_init": 1.0,
            "swish_beta_learnable": False,
        },
    )

    train_start = perf_counter()
    summary = train_or_load_summary(cfg, f"test2/capacity/depth_{depth}", f"Test2 depth={depth}")
    train_s = perf_counter() - train_start

    saved_config, model, _normalizer = load_trained_artifacts(summary)
    params = int(sum(p.numel() for p in model.parameters()))
    depth_rows.append(
        {
            "depth": int(depth),
            "best_val_loss": float(summary["best_val_loss"]),
            "train_s": float(train_s),
            "params": int(params),
            "hidden_size": int(saved_config.get("model", {}).get("hidden_size", 128)),
        }
    )


width_df = pd.DataFrame(width_rows).sort_values("width").reset_index(drop=True)
depth_df = pd.DataFrame(depth_rows).sort_values("depth").reset_index(drop=True)

display(width_df)
display(depth_df)

# Width sweep plot (val loss only)
fig, ax1 = plt.subplots(figsize=(10, 5.5))
line_val_w, = ax1.plot(
    width_df["width"],
    width_df["best_val_loss"],
    marker="o",
    linewidth=1.5,
    label="Best val loss",
)
ax1.set_xlabel("Hidden size (width)")
ax1.set_ylabel("Best val loss")
ax1.set_title("Test 2A: Width sweep (depth fixed at 3)")
ax1.grid(alpha=0.3)

best_width_row = width_df.loc[width_df["best_val_loss"].idxmin()]
line_knee_w = ax1.axvline(
    float(best_width_row["width"]),
    color="gray",
    linestyle=":",
    linewidth=1,
    label=f"Knee width = {int(best_width_row['width'])}",
)

ax1.legend(
    handles=[line_val_w, line_knee_w],
    loc="upper left",
    fontsize=8,
    framealpha=0.9,
)

ann_width = make_plot_annotation(
    model_type="MLP pointwise",
    arch_dict={"layers": 3, "width sweep": list(width_df["width"].astype(int)), "activation": "Swish beta=1.0 fixed"},
    train_dict={
        "epochs": CAPACITY_EPOCHS,
        "seed": SEED,
        "batch": BASE_MLP_CONFIG.get("train", {}).get("batch_size"),
        "lr": BASE_MLP_CONFIG.get("train", {}).get("lr", BASE_MLP_CONFIG.get("train", {}).get("learning_rate")),
        "wd": BASE_MLP_CONFIG.get("train", {}).get("weight_decay"),
        "clip": BASE_MLP_CONFIG.get("train", {}).get("grad_clip"),
        "patience": BASE_MLP_CONFIG.get("train", {}).get("early_stopping_patience"),
    },
    data_split_dict=BENCH_OPS,
    loss_dict={
        "T_weight": BASE_MLP_CONFIG.get("loss", {}).get("T_weight", 1.0),
        "bc_V_weight": BASE_MLP_CONFIG.get("loss", {}).get("bc_V_weight", 1.0),
    },
)
add_annotation_box(ax1, ann_width)
fig.tight_layout()
plt.show()

# Depth sweep plot (val loss only)
fig, ax1 = plt.subplots(figsize=(10, 5.5))
line_val_d, = ax1.plot(
    depth_df["depth"],
    depth_df["best_val_loss"],
    marker="o",
    linewidth=1.5,
    label="Best val loss",
)
ax1.set_xlabel("Hidden layers (depth)")
ax1.set_ylabel("Best val loss")
ax1.set_title("Test 2B: Depth sweep (width fixed at 128)")
ax1.grid(alpha=0.3)

best_depth_row = depth_df.loc[depth_df["best_val_loss"].idxmin()]
line_knee_d = ax1.axvline(
    float(best_depth_row["depth"]),
    color="gray",
    linestyle=":",
    linewidth=1,
    label=f"Knee depth = {int(best_depth_row['depth'])}",
)

ax1.legend(
    handles=[line_val_d, line_knee_d],
    loc="upper left",
    fontsize=8,
    framealpha=0.9,
)

ann_depth = make_plot_annotation(
    model_type="MLP pointwise",
    arch_dict={"depth sweep": list(depth_df["depth"].astype(int)), "width": 128, "activation": "Swish beta=1.0 fixed"},
    train_dict={
        "epochs": CAPACITY_EPOCHS,
        "seed": SEED,
        "batch": BASE_MLP_CONFIG.get("train", {}).get("batch_size"),
        "lr": BASE_MLP_CONFIG.get("train", {}).get("lr", BASE_MLP_CONFIG.get("train", {}).get("learning_rate")),
        "wd": BASE_MLP_CONFIG.get("train", {}).get("weight_decay"),
        "clip": BASE_MLP_CONFIG.get("train", {}).get("grad_clip"),
        "patience": BASE_MLP_CONFIG.get("train", {}).get("early_stopping_patience"),
    },
    data_split_dict=BENCH_OPS,
    loss_dict={
        "T_weight": BASE_MLP_CONFIG.get("loss", {}).get("T_weight", 1.0),
        "bc_V_weight": BASE_MLP_CONFIG.get("loss", {}).get("bc_V_weight", 1.0),
    },
)
add_annotation_box(ax1, ann_depth)
fig.tight_layout()
plt.show()

print("Test 2 complete.")
print(f"Best width by val loss: width={int(best_width_row['width'])}, val_loss={best_width_row['best_val_loss']:.6e}")
print(f"Best depth by val loss: depth={int(best_depth_row['depth'])}, val_loss={best_depth_row['best_val_loss']:.6e}")
print(f"Progress status: done={_progress_state['done']} / total={TOTAL_RUNS}")

## Test 2C - True vs Predicted for MLP (width=128, depth=3)

This plot uses the model artifact from the width sweep entry with hidden size 128 and 3 hidden layers, then evaluates it on the validation OP set.

In [ ]:
# -----------------------------------------------------
# Test 2C: True vs Predicted on validation data
# Model: width=128, depth=3
# -----------------------------------------------------

selected_tag = "test2/capacity/width_128"
selected_dir = latest_artifact_dir(selected_tag)
if selected_dir is None:
    raise RuntimeError(
        "No artifact found for width=128, depth=3. Run the Test 2 capacity cell first."
    )

selected_summary = {
    "config_path": str(selected_dir / "config.yaml"),
    "best_ckpt": str(selected_dir / "best.pt"),
    "normalizer": str(selected_dir / "normalizer.json"),
    "n_sensors": 363,
}

sel_cfg, sel_model, sel_norm = load_trained_artifacts(selected_summary)

val_pairs = collect_pointwise_predictions(
    sel_model,
    BENCH_VAL_OPS,
    sel_norm,
    subsample_time=int(sel_cfg.get("data", {}).get("subsample_time", 50)),
    ts_extrapolation=TS_EXTRAPOLATION,
    device=DEVICE,
)

metric_T = metric_bundle_pointwise(val_pairs["T"]["y_true"], val_pairs["T"]["y_pred"])
metric_V = metric_bundle_pointwise(val_pairs["bc_V"]["y_true"], val_pairs["bc_V"]["y_pred"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_specs = [
    (axes[0], "T", metric_T),
    (axes[1], "bc_V", metric_V),
]

for ax, target_name, metric in plot_specs:
    y_true = val_pairs[target_name]["y_true"]
    y_pred = val_pairs[target_name]["y_pred"]

    if len(y_true) == 0:
        ax.set_title(f"{target_name}: no validation points")
        continue

    n_plot = min(25000, len(y_true))
    if len(y_true) > n_plot:
        idx = np.random.RandomState(SEED).choice(len(y_true), n_plot, replace=False)
        y_true_plot = y_true[idx]
        y_pred_plot = y_pred[idx]
    else:
        y_true_plot = y_true
        y_pred_plot = y_pred

    ax.scatter(y_true_plot, y_pred_plot, s=3, alpha=0.18)
    lo = float(min(np.min(y_true_plot), np.min(y_pred_plot)))
    hi = float(max(np.max(y_true_plot), np.max(y_pred_plot)))
    ax.plot([lo, hi], [lo, hi], "r--", linewidth=1)
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")
    ax.set_title(
        f"{target_name} | MAE={metric['mae']:.3e}, RMSE={np.sqrt(metric['mse']):.3e}, R2={metric['r2']:.4f}"
    )

ann_scatter = make_plot_annotation(
    model_type="MLP pointwise",
    arch_dict={"layers": 3, "hidden_size": 128, "activation": "Swish beta=1.0 fixed"},
    train_dict={
        "epochs": CAPACITY_EPOCHS,
        "seed": SEED,
        "batch": sel_cfg.get("train", {}).get("batch_size"),
        "lr": sel_cfg.get("train", {}).get("lr", sel_cfg.get("train", {}).get("learning_rate")),
        "wd": sel_cfg.get("train", {}).get("weight_decay"),
        "clip": sel_cfg.get("train", {}).get("grad_clip"),
        "patience": sel_cfg.get("train", {}).get("early_stopping_patience"),
    },
    data_split_dict={"train": BENCH_TRAIN_OPS, "val": BENCH_VAL_OPS, "test": BENCH_TEST_OPS},
    loss_dict={
        "T_weight": sel_cfg.get("loss", {}).get("T_weight", 1.0),
        "bc_V_weight": sel_cfg.get("loss", {}).get("bc_V_weight", 1.0),
    },
)
add_annotation_box(axes[1], ann_scatter)

fig.suptitle("Validation true vs predicted: MLP width=128, depth=3")
fig.tight_layout()
plt.show()

print(f"Validation points: {val_pairs['n_points']}")
print(f"T metrics: MAE={metric_T['mae']:.6e}, RMSE={np.sqrt(metric_T['mse']):.6e}, R2={metric_T['r2']:.6f}")
print(f"bc_V metrics: MAE={metric_V['mae']:.6e}, RMSE={np.sqrt(metric_V['mse']):.6e}, R2={metric_V['r2']:.6f}")

## Section 3 - Activation Study (MLP only)

This section isolates the Swish activation beta on real OP data.

What changes in this test:
- Sweep `swish_beta_init in BETA_VALUES`
- Keep `swish_beta_learnable = False`

What stays fixed:
- Model architecture: `n_hidden_layers = 3`, `hidden_size = 128`
- Split: train = OP01, OP02, OP08; val = OP09, OP10; test = OP11, OP12
- Epoch budget per run: `BETA_EPOCHS`
- Same seed, same weighted-MSE loss weights

Output:
- Validation loss vs Swish beta, with one consistent configuration annotation box

Note:
- All other benchmark sections keep `beta = 1.0` fixed unless explicitly stated otherwise.

In [ ]:
# -------------------------------------------
# Section 3: Activation (Swish beta) study
# -------------------------------------------

# Fallback helpers if this section is run independently.
if "make_plot_annotation" not in globals():
    def make_plot_annotation(
        model_type: str,
        arch_dict: dict,
        train_dict: dict,
        data_split_dict: dict,
        loss_dict: dict,
    ) -> str:
        arch_text = ", ".join(f"{k}={v}" for k, v in arch_dict.items())
        train_text = ", ".join(f"{k}={v}" for k, v in train_dict.items())
        data_text = (
            f"train={data_split_dict['train']} | val={data_split_dict['val']} | test={data_split_dict['test']}"
        )
        loss_text = ", ".join(f"{k}={v}" for k, v in loss_dict.items())
        return (
            f"Model: {model_type}\n"
            f"Arch: {arch_text}\n"
            f"Train: {train_text}\n"
            f"Data: {data_text}\n"
            f"Loss: weighted MSE ({loss_text})"
        )

if "add_annotation_box" not in globals():
    def add_annotation_box(ax, text: str) -> None:
        ax.text(
            0.98,
            0.02,
            text,
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=7,
            bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "gray"},
        )

beta_rows: list[dict] = []
for beta in BETA_VALUES:
    cfg = build_benchmark_config(
        BASE_MLP_CONFIG,
        model_type="mlp_pointwise",
        train_ops=BENCH_TRAIN_OPS,
        val_ops=BENCH_VAL_OPS,
        test_ops=BENCH_TEST_OPS,
        epochs=BETA_EPOCHS,
        output_tag=f"test3/activation/beta_{beta}",
        model_overrides={
            "n_hidden_layers": 3,
            "hidden_size": 128,
            "swish_beta_init": float(beta),
            "swish_beta_learnable": False,
        },
    )

    train_start = perf_counter()
    summary = train_or_load_summary(cfg, f"test3/activation/beta_{beta}", f"Section3 beta={beta}")
    train_s = perf_counter() - train_start

    saved_config, model, _normalizer = load_trained_artifacts(summary)
    params = int(sum(p.numel() for p in model.parameters()))
    beta_rows.append(
        {
            "beta": float(beta),
            "best_val_loss": float(summary["best_val_loss"]),
            "train_s": float(train_s),
            "params": int(params),
            "layers": int(saved_config.get("model", {}).get("n_hidden_layers", 3)),
            "hidden_size": int(saved_config.get("model", {}).get("hidden_size", 128)),
        }
    )

beta_df = pd.DataFrame(beta_rows).sort_values("beta").reset_index(drop=True)
display(beta_df)

fig, ax = plt.subplots(figsize=(10, 5.5))
line_val, = ax.plot(
    beta_df["beta"],
    beta_df["best_val_loss"],
    marker="o",
    linewidth=1.5,
    label="Best val loss",
)
ax.set_xlabel("Swish beta")
ax.set_ylabel("Best val loss")
ax.set_title("Section 3: Activation sweep (MLP, fixed architecture)")
ax.grid(alpha=0.3)

best_beta_row = beta_df.loc[beta_df["best_val_loss"].idxmin()]
line_best = ax.axvline(
    float(best_beta_row["beta"]),
    color="gray",
    linestyle=":",
    linewidth=1,
    label=f"Best beta = {best_beta_row['beta']:.2f}",
)

ax.legend(handles=[line_val, line_best], loc="upper right", fontsize=8, framealpha=0.9)

ann_beta = make_plot_annotation(
    model_type="MLP pointwise",
    arch_dict={
        "layers": 3,
        "width": 128,
        "activation": f"Swish beta sweep={list(beta_df['beta'])}, learnable=False",
    },
    train_dict={
        "epochs": BETA_EPOCHS,
        "seed": SEED,
        "batch": BASE_MLP_CONFIG.get("train", {}).get("batch_size"),
        "lr": BASE_MLP_CONFIG.get("train", {}).get("lr", BASE_MLP_CONFIG.get("train", {}).get("learning_rate")),
        "wd": BASE_MLP_CONFIG.get("train", {}).get("weight_decay"),
        "clip": BASE_MLP_CONFIG.get("train", {}).get("grad_clip"),
        "patience": BASE_MLP_CONFIG.get("train", {}).get("early_stopping_patience"),
    },
    data_split_dict=BENCH_OPS,
    loss_dict={
        "T_weight": BASE_MLP_CONFIG.get("loss", {}).get("T_weight", 1.0),
        "bc_V_weight": BASE_MLP_CONFIG.get("loss", {}).get("bc_V_weight", 1.0),
    },
)
ax.text(
    0.98,
    0.78,
    ann_beta,
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=7,
    bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "gray"},
)
fig.tight_layout()
plt.show()

print("Section 3 complete.")
print(f"Best beta by val loss: beta={best_beta_row['beta']:.2f}, val_loss={best_beta_row['best_val_loss']:.6e}")
if "_progress_state" in globals():
    print(f"Progress status: done={_progress_state['done']} / total={TOTAL_RUNS}")
print("All other sections keep beta=1.0 fixed unless explicitly varied.")

## Section 3B - Layer Sweep (Single Fixed Beta) + Best Validation Plot

This follow-up removes the layer-beta grid and uses a single fixed beta for all runs.

What varies:
- `n_hidden_layers` over `CAPACITY_DEPTHS`

What stays fixed:
- Width fixed to the best known width if available (fallback: 128)
- Beta fixed for all runs: `swish_beta_init = 1.0`
- `swish_beta_learnable = False`
- Same train/val/test OP split and weighted-MSE loss

Outputs:
- Table of `n_hidden_layers` vs validation loss
- Validation true-vs-predicted scatter for the best layer configuration

In [ ]:
# ------------------------------------------------------------------
# Section 3B: Layer sweep table (single fixed beta for all runs)
# ------------------------------------------------------------------

fixed_width = int(best_width_row["width"]) if "best_width_row" in globals() else 128
fixed_beta = 1.0
print(f"Fixed settings for sweep: width={fixed_width}, beta={fixed_beta:.2f}")

layer_rows: list[dict] = []
for depth in CAPACITY_DEPTHS:
    tag = f"test3b/layer_only/depth_{depth}_w{fixed_width}_b{fixed_beta:.1f}"
    cfg = build_benchmark_config(
        BASE_MLP_CONFIG,
        model_type="mlp_pointwise",
        train_ops=BENCH_TRAIN_OPS,
        val_ops=BENCH_VAL_OPS,
        test_ops=BENCH_TEST_OPS,
        epochs=CAPACITY_EPOCHS,
        output_tag=tag,
        model_overrides={
            "n_hidden_layers": int(depth),
            "hidden_size": int(fixed_width),
            "swish_beta_init": float(fixed_beta),
            "swish_beta_learnable": False,
        },
    )

    train_start = perf_counter()
    summary = train_or_load_summary(cfg, tag, f"Section3B depth={depth}")
    train_s = perf_counter() - train_start

    saved_config, model, _normalizer = load_trained_artifacts(summary)
    params = int(sum(p.numel() for p in model.parameters()))
    layer_rows.append(
        {
            "n_hidden_layers": int(depth),
            "best_val_loss": float(summary["best_val_loss"]),
            "train_s": float(train_s),
            "params": int(params),
            "hidden_size": int(saved_config.get("model", {}).get("hidden_size", fixed_width)),
            "beta": float(saved_config.get("model", {}).get("swish_beta_init", fixed_beta)),
        }
    )

layer_df = pd.DataFrame(layer_rows).sort_values(["n_hidden_layers"]).reset_index(drop=True)
print("Layer sweep results (single fixed beta):")
display(layer_df)

best_row = layer_df.loc[layer_df["best_val_loss"].idxmin()]
best_depth = int(best_row["n_hidden_layers"])

best_tag = f"test3b/layer_only/depth_{best_depth}_w{fixed_width}_b{fixed_beta:.1f}"
best_dir = latest_artifact_dir(best_tag)
if best_dir is None:
    raise RuntimeError("No artifact found for best layer model. Run this cell from top.")

best_summary = {
    "config_path": str(best_dir / "config.yaml"),
    "best_ckpt": str(best_dir / "best.pt"),
    "normalizer": str(best_dir / "normalizer.json"),
    "n_sensors": 363,
}
best_cfg, best_model, best_norm = load_trained_artifacts(best_summary)

val_pairs = collect_pointwise_predictions(
    best_model,
    BENCH_VAL_OPS,
    best_norm,
    subsample_time=int(best_cfg.get("data", {}).get("subsample_time", 50)),
    ts_extrapolation=TS_EXTRAPOLATION,
    device=DEVICE,
)

metric_T = metric_bundle_pointwise(val_pairs["T"]["y_true"], val_pairs["T"]["y_pred"])
metric_V = metric_bundle_pointwise(val_pairs["bc_V"]["y_true"], val_pairs["bc_V"]["y_pred"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_specs = [
    (axes[0], "T", metric_T),
    (axes[1], "bc_V", metric_V),
]

for ax, target_name, metric in plot_specs:
    y_true = val_pairs[target_name]["y_true"]
    y_pred = val_pairs[target_name]["y_pred"]

    if len(y_true) == 0:
        ax.set_title(f"{target_name}: no validation points")
        continue

    n_plot = min(25000, len(y_true))
    if len(y_true) > n_plot:
        idx = np.random.RandomState(SEED).choice(len(y_true), n_plot, replace=False)
        y_true_plot = y_true[idx]
        y_pred_plot = y_pred[idx]
    else:
        y_true_plot = y_true
        y_pred_plot = y_pred

    ax.scatter(y_true_plot, y_pred_plot, s=3, alpha=0.18)
    lo = float(min(np.min(y_true_plot), np.min(y_pred_plot)))
    hi = float(max(np.max(y_true_plot), np.max(y_pred_plot)))
    ax.plot([lo, hi], [lo, hi], "r--", linewidth=1)
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")
    ax.set_title(
        f"{target_name} | MAE={metric['mae']:.3e}, RMSE={np.sqrt(metric['mse']):.3e}, R2={metric['r2']:.4f}"
    )

best_ann = make_plot_annotation(
    model_type="MLP pointwise",
    arch_dict={
        "layers": best_depth,
        "width": fixed_width,
        "activation": f"Swish beta={fixed_beta:.1f}, learnable=False",
    },
    train_dict={
        "epochs": CAPACITY_EPOCHS,
        "seed": SEED,
        "batch": best_cfg.get("train", {}).get("batch_size"),
        "lr": best_cfg.get("train", {}).get("lr", best_cfg.get("train", {}).get("learning_rate")),
        "wd": best_cfg.get("train", {}).get("weight_decay"),
        "clip": best_cfg.get("train", {}).get("grad_clip"),
        "patience": best_cfg.get("train", {}).get("early_stopping_patience"),
    },
    data_split_dict={"train": BENCH_TRAIN_OPS, "val": BENCH_VAL_OPS, "test": BENCH_TEST_OPS},
    loss_dict={
        "T_weight": best_cfg.get("loss", {}).get("T_weight", 1.0),
        "bc_V_weight": best_cfg.get("loss", {}).get("bc_V_weight", 1.0),
    },
)

axes[1].text(
    0.98,
    0.98,
    best_ann,
    transform=axes[1].transAxes,
    ha="right",
    va="top",
    fontsize=7,
    bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "gray"},
)

fig.suptitle(
    f"Best validation model from layer sweep: depth={best_depth}, width={fixed_width}, beta={fixed_beta:.1f}"
)
fig.tight_layout()
plt.show()

print(f"Best by validation loss: depth={best_depth}, beta={fixed_beta:.1f}, val_loss={best_row['best_val_loss']:.6e}")
print(f"Validation points: {val_pairs['n_points']}")
print(f"T metrics: MAE={metric_T['mae']:.6e}, RMSE={np.sqrt(metric_T['mse']):.6e}, R2={metric_T['r2']:.6f}")
print(f"bc_V metrics: MAE={metric_V['mae']:.6e}, RMSE={np.sqrt(metric_V['mse']):.6e}, R2={metric_V['r2']:.6f}")

## Section 4 - History-Length Study (Recurrent only)

This section isolates memory length for the recurrent model on real OP data.

What changes in this test:
- Sweep `history_length` over `HISTORY_K_VALUES` (steps)

What stays fixed:
- Same train/val/test split and weighted-MSE setup
- Same seed and recurrent backbone settings
- Recurrent training uses `batch_size = 1`
- Epoch budget per history length: `HISTORY_EPOCHS`

Outputs:
- MAE vs history length for `T` and `bc_V`
- Secondary x-axis in seconds using `TIME_DELTA_S = 0.1`
- Recommended history length (`k`) from the benchmark output

In [18]:
# -------------------------------------------------------------
# Section 4: Recurrent history-length benchmark (steps -> sec)
# -------------------------------------------------------------

# Fallback annotation helpers if this section is run independently.
if "make_plot_annotation" not in globals():
    def make_plot_annotation(
        model_type: str,
        arch_dict: dict,
        train_dict: dict,
        data_split_dict: dict,
        loss_dict: dict,
    ) -> str:
        arch_text = ", ".join(f"{k}={v}" for k, v in arch_dict.items())
        train_text = ", ".join(f"{k}={v}" for k, v in train_dict.items())
        data_text = (
            f"train={data_split_dict['train']} | val={data_split_dict['val']} | test={data_split_dict['test']}"
        )
        loss_text = ", ".join(f"{k}={v}" for k, v in loss_dict.items())
        return (
            f"Model: {model_type}\n"
            f"Arch: {arch_text}\n"
            f"Train: {train_text}\n"
            f"Data: {data_text}\n"
            f"Loss: weighted MSE ({loss_text})"
        )

if "add_annotation_box" not in globals():
    def add_annotation_box(ax, text: str) -> None:
        ax.text(
            0.98,
            0.02,
            text,
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=7,
            bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "gray"},
        )

history_cfg = build_benchmark_config(
    BASE_RECURRENT_CONFIG,
    model_type="recurrent",
    train_ops=BENCH_TRAIN_OPS,
    val_ops=BENCH_VAL_OPS,
    test_ops=BENCH_TEST_OPS,
    epochs=HISTORY_EPOCHS,
    output_tag="test4/history_sweep",
)
history_cfg.setdefault("train", {})
history_cfg["train"]["batch_size"] = 1

# Route nested sweep progress into the one global benchmark progress counter.
history_progress = make_progress_proxy()

history_df = benchmark_history_lengths(
    history_cfg,
    k_values=HISTORY_K_VALUES,
    epochs_per_k=HISTORY_EPOCHS,
    max_sensors=10,
    device=DEVICE,
    progress_cb=history_progress,
)

if history_df.empty:
    raise RuntimeError("History benchmark returned no rows.")

history_df = history_df.sort_values("k").reset_index(drop=True)
display(history_df)

fig, ax = plt.subplots(figsize=(11, 5.5))
line_t, = ax.plot(history_df["k"], history_df["mae_T"], marker="o", linewidth=1.5, label="MAE_T")
line_v, = ax.plot(history_df["k"], history_df["mae_bc_V"], marker="s", linewidth=1.5, label="MAE_bc_V")
ax.set_xlabel("History length k (steps)")
ax.set_ylabel("MAE")
ax.set_title("Section 4: Recurrent history-length sweep")
ax.grid(alpha=0.3)

# Secondary axis: seconds = steps * TIME_DELTA_S
secax = ax.secondary_xaxis(
    "top",
    functions=(lambda x: x * TIME_DELTA_S, lambda s: s / TIME_DELTA_S),
)
secax.set_xlabel("Lookback (seconds)")

recommended_rows = history_df[history_df.get("recommended", False) == True]
if len(recommended_rows) > 0:
    recommended_row = recommended_rows.iloc[0]
else:
    # Fallback if recommended column is absent or all False
    recommended_idx = (history_df["mae_T"] + history_df["mae_bc_V"]).idxmin()
    recommended_row = history_df.loc[recommended_idx]

recommended_k = int(recommended_row["k"])
line_k = ax.axvline(
    recommended_k,
    color="gray",
    linestyle=":",
    linewidth=1,
    label=f"Recommended k = {recommended_k}",
)

ax.legend(handles=[line_t, line_v, line_k], loc="upper right", fontsize=8, framealpha=0.9)

lookback_median = float(recommended_row.get("lookback_seconds_median", recommended_k * TIME_DELTA_S))
lookback_min = float(recommended_row.get("lookback_seconds_min", recommended_k * TIME_DELTA_S))
lookback_max = float(recommended_row.get("lookback_seconds_max", recommended_k * TIME_DELTA_S))

ann_history = make_plot_annotation(
    model_type="Recurrent (GRU)",
    arch_dict={
        "history sweep": [int(k) for k in history_df["k"].tolist()],
        "recommended_k": recommended_k,
        "lookback_s": f"median={lookback_median:.2f}, min={lookback_min:.2f}, max={lookback_max:.2f}",
    },
    train_dict={
        "epochs_per_k": HISTORY_EPOCHS,
        "seed": SEED,
        "batch": 1,
        "lr": history_cfg.get("train", {}).get("lr", history_cfg.get("train", {}).get("learning_rate")),
        "wd": history_cfg.get("train", {}).get("weight_decay"),
        "clip": history_cfg.get("train", {}).get("grad_clip"),
        "patience": history_cfg.get("train", {}).get("early_stopping_patience"),
    },
    data_split_dict=BENCH_OPS,
    loss_dict={
        "T_weight": history_cfg.get("loss", {}).get("T_weight", 1.0),
        "bc_V_weight": history_cfg.get("loss", {}).get("bc_V_weight", 1.0),
    },
)
add_annotation_box(ax, ann_history)

fig.tight_layout()
plt.show()

print("Section 4 complete.")
print("Sanity check: length=20 -> 2.0 s, length=50 -> 5.0 s (TIME_DELTA_S = 0.1)")
print(f"Recommended k: {recommended_k}")
print(f"Recommended lookback seconds: median={lookback_median:.3f}, min={lookback_min:.3f}, max={lookback_max:.3f}")

Full benchmark suite:   1%|▏         | 1/67 [16:27<18:06:46, 987.97s/it, k=1]

KeyboardInterrupt: 

## Section 5 - MLP Hyperparameter Optimization (80/10/10 split, no OP19)

This benchmark performs MLP-only hyperparameter optimization on real OP bundles.

Rules for this section:
- Recurrent model is skipped (runtime).
- OP19 is excluded from this optimization split.
- Deterministic 80/10/10 split is built from the remaining OPs.
- Search is bounded (random sampled trials) to stay CPU-runnable.

Outputs:
- Trial table sorted by validation score
- Best hyperparameter set
- Final test metrics on the 10% test split

In [19]:
# -----------------------------------------------------------------
# Section 5: MLP hyperparameter optimization (80/10/10, no OP19)
# -----------------------------------------------------------------

import itertools
import math

# Fallback annotation helpers if this section is run independently.
if "make_plot_annotation" not in globals():
    def make_plot_annotation(
        model_type: str,
        arch_dict: dict,
        train_dict: dict,
        data_split_dict: dict,
        loss_dict: dict,
    ) -> str:
        arch_text = ", ".join(f"{k}={v}" for k, v in arch_dict.items())
        train_text = ", ".join(f"{k}={v}" for k, v in train_dict.items())
        data_text = (
            f"train={data_split_dict['train']} | val={data_split_dict['val']} | test={data_split_dict['test']}"
        )
        loss_text = ", ".join(f"{k}={v}" for k, v in loss_dict.items())
        return (
            f"Model: {model_type}\n"
            f"Arch: {arch_text}\n"
            f"Train: {train_text}\n"
            f"Data: {data_text}\n"
            f"Loss: weighted MSE ({loss_text})"
        )

if "add_annotation_box" not in globals():
    def add_annotation_box(ax, text: str) -> None:
        ax.text(
            0.98,
            0.02,
            text,
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=7,
            bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "gray"},
        )

# Real OP pool (from available cached bundles), excluding OP19 by request.
op_pool_all = [
    "OP01", "OP02", "OP03", "OP08", "OP09", "OP10", "OP11", "OP12",
    "OP13", "OP14", "OP15", "OP16", "OP19",
]
op_pool = [op for op in op_pool_all if op != "OP19"]

rng = np.random.RandomState(SEED)
op_shuffled = op_pool.copy()
rng.shuffle(op_shuffled)

n_total = len(op_shuffled)
n_train = int(round(0.8 * n_total))
n_val = int(round(0.1 * n_total))

# Keep at least one OP in val and test.
if n_train > n_total - 2:
    n_train = n_total - 2
if n_val < 1:
    n_val = 1
if n_train + n_val > n_total - 1:
    n_val = max(1, n_total - 1 - n_train)

opt_train_ops = op_shuffled[:n_train]
opt_val_ops = op_shuffled[n_train:n_train + n_val]
opt_test_ops = op_shuffled[n_train + n_val:]

print("Section 5 split (OP19 excluded):")
print(f"Train ({len(opt_train_ops)}): {opt_train_ops}")
print(f"Val   ({len(opt_val_ops)}): {opt_val_ops}")
print(f"Test  ({len(opt_test_ops)}): {opt_test_ops}")

if len(opt_val_ops) == 0 or len(opt_test_ops) == 0:
    raise RuntimeError("Invalid 80/10/10 split result; val/test cannot be empty.")

# Bounded search space for CPU-runnable optimization.
HP_N_HIDDEN_LAYERS = [2, 3, 4, 5]
HP_HIDDEN_SIZE = [64, 128, 256, 512]
HP_SWISH_BETA = [0.5, 1.0, 1.5, 2.0]
HP_LR = [5e-4, 1e-3, 2e-3]
HP_WEIGHT_DECAY = [0.0, 1e-6, 1e-5]
HP_BATCH_SIZE = [2048, 4096, 8192]
HP_GRAD_CLIP = [0.5, 1.0, 2.0]
HP_PATIENCE = [6, 10, 14]
HP_SUBSAMPLE = [25, 50, 75]

OPT_EPOCHS = 6
OPT_TRIALS = 24

all_combos = list(itertools.product(
    HP_N_HIDDEN_LAYERS,
    HP_HIDDEN_SIZE,
    HP_SWISH_BETA,
    HP_LR,
    HP_WEIGHT_DECAY,
    HP_BATCH_SIZE,
    HP_GRAD_CLIP,
    HP_PATIENCE,
    HP_SUBSAMPLE,
))

if OPT_TRIALS > len(all_combos):
    OPT_TRIALS = len(all_combos)

chosen_idx = rng.choice(len(all_combos), size=OPT_TRIALS, replace=False)

trial_rows: list[dict] = []
for trial_i, combo_idx in enumerate(chosen_idx, start=1):
    (
        hp_layers,
        hp_width,
        hp_beta,
        hp_lr,
        hp_wd,
        hp_bs,
        hp_clip,
        hp_patience,
        hp_subsample,
    ) = all_combos[int(combo_idx)]

    trial_tag = f"test5/mlp_opt_80_10_10/trial_{trial_i:03d}"

    cfg = build_benchmark_config(
        BASE_MLP_CONFIG,
        model_type="mlp_pointwise",
        train_ops=opt_train_ops,
        val_ops=opt_val_ops,
        test_ops=opt_test_ops,
        epochs=OPT_EPOCHS,
        output_tag=trial_tag,
        model_overrides={
            "n_hidden_layers": int(hp_layers),
            "hidden_size": int(hp_width),
            "swish_beta_init": float(hp_beta),
            "swish_beta_learnable": False,
        },
    )

    cfg.setdefault("train", {})
    cfg.setdefault("data", {})
    cfg["train"]["lr"] = float(hp_lr)
    cfg["train"]["weight_decay"] = float(hp_wd)
    cfg["train"]["batch_size"] = int(hp_bs)
    cfg["train"]["grad_clip"] = float(hp_clip)
    cfg["train"]["early_stopping_patience"] = int(hp_patience)
    cfg["data"]["subsample_time"] = int(hp_subsample)

    train_t0 = perf_counter()
    summary = train_or_load_summary(cfg, trial_tag, f"Section5 trial {trial_i}/{OPT_TRIALS}")
    train_s = perf_counter() - train_t0

    saved_cfg, model, normalizer = load_trained_artifacts(summary)
    params = int(sum(p.numel() for p in model.parameters()))

    val_t0 = perf_counter()
    val_pairs = collect_pointwise_predictions(
        model,
        opt_val_ops,
        normalizer,
        subsample_time=int(saved_cfg.get("data", {}).get("subsample_time", hp_subsample)),
        ts_extrapolation=TS_EXTRAPOLATION,
        device=DEVICE,
    )
    val_infer_s = max(perf_counter() - val_t0, 1e-12)

    m_t = metric_bundle_pointwise(val_pairs["T"]["y_true"], val_pairs["T"]["y_pred"])
    m_v = metric_bundle_pointwise(val_pairs["bc_V"]["y_true"], val_pairs["bc_V"]["y_pred"])

    rmse_t = math.sqrt(float(m_t["mse"]))
    rmse_v = math.sqrt(float(m_v["mse"]))
    val_score = rmse_t + rmse_v

    trial_rows.append({
        "trial": int(trial_i),
        "layers": int(hp_layers),
        "width": int(hp_width),
        "beta": float(hp_beta),
        "lr": float(hp_lr),
        "weight_decay": float(hp_wd),
        "batch_size": int(hp_bs),
        "grad_clip": float(hp_clip),
        "patience": int(hp_patience),
        "subsample_time": int(hp_subsample),
        "params": int(params),
        "best_val_loss": float(summary["best_val_loss"]),
        "val_rmse_T": float(rmse_t),
        "val_rmse_bc_V": float(rmse_v),
        "val_score": float(val_score),
        "val_r2_T": float(m_t["r2"]),
        "val_r2_bc_V": float(m_v["r2"]),
        "train_s": float(train_s),
        "val_infer_pts_per_s": float(val_pairs["n_points"] / val_infer_s),
        "trial_tag": trial_tag,
    })

trials_df = pd.DataFrame(trial_rows).sort_values("val_score").reset_index(drop=True)
display(trials_df.head(10))

best_trial = trials_df.iloc[0]
best_trial_tag = str(best_trial["trial_tag"])
best_trial_dir = latest_artifact_dir(best_trial_tag)
if best_trial_dir is None:
    raise RuntimeError(f"Could not find artifact dir for best trial tag: {best_trial_tag}")

best_summary = {
    "config_path": str(best_trial_dir / "config.yaml"),
    "best_ckpt": str(best_trial_dir / "best.pt"),
    "normalizer": str(best_trial_dir / "normalizer.json"),
    "n_sensors": 363,
}
best_cfg_opt, best_model_opt, best_norm_opt = load_trained_artifacts(best_summary)

test_t0 = perf_counter()
test_pairs = collect_pointwise_predictions(
    best_model_opt,
    opt_test_ops,
    best_norm_opt,
    subsample_time=int(best_cfg_opt.get("data", {}).get("subsample_time", 50)),
    ts_extrapolation=TS_EXTRAPOLATION,
    device=DEVICE,
)
test_infer_s = max(perf_counter() - test_t0, 1e-12)

test_m_t = metric_bundle_pointwise(test_pairs["T"]["y_true"], test_pairs["T"]["y_pred"])
test_m_v = metric_bundle_pointwise(test_pairs["bc_V"]["y_true"], test_pairs["bc_V"]["y_pred"])

test_table = pd.DataFrame([
    {
        "model": "Best MLP (opt)",
        "MAE_T": float(test_m_t["mae"]),
        "RMSE_T": float(math.sqrt(test_m_t["mse"])),
        "R2_T": float(test_m_t["r2"]),
        "MAE_bc_V": float(test_m_v["mae"]),
        "RMSE_bc_V": float(math.sqrt(test_m_v["mse"])),
        "R2_bc_V": float(test_m_v["r2"]),
        "params": int(best_trial["params"]),
        "infer_pts_per_s": float(test_pairs["n_points"] / test_infer_s),
    }
])

display(test_table)

# Compact plot of top trial validation scores.
top_k = min(10, len(trials_df))
plot_df = trials_df.head(top_k).copy()
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(plot_df["trial"].astype(str), plot_df["val_score"])
ax.set_xlabel("Trial")
ax.set_ylabel("Validation score (RMSE_T + RMSE_bc_V)")
ax.set_title("Section 5: Top MLP trials (lower is better)")
ax.grid(axis="y", alpha=0.3)

ann_opt = make_plot_annotation(
    model_type="MLP pointwise (optimization)",
    arch_dict={
        "search_layers": HP_N_HIDDEN_LAYERS,
        "search_width": HP_HIDDEN_SIZE,
        "search_beta": HP_SWISH_BETA,
        "best_trial": int(best_trial["trial"]),
    },
    train_dict={
        "epochs": OPT_EPOCHS,
        "trials": OPT_TRIALS,
        "seed": SEED,
        "batch_search": HP_BATCH_SIZE,
        "lr_search": HP_LR,
    },
    data_split_dict={"train": opt_train_ops, "val": opt_val_ops, "test": opt_test_ops},
    loss_dict={
        "T_weight": best_cfg_opt.get("loss", {}).get("T_weight", 1.0),
        "bc_V_weight": best_cfg_opt.get("loss", {}).get("bc_V_weight", 1.0),
    },
)
add_annotation_box(ax, ann_opt)
fig.tight_layout()
plt.show()

print("Section 5 complete (MLP only).")
print("OP19 usage: excluded from train/val/test in this section.")
print(f"Best trial tag: {best_trial_tag}")
print(
    f"Best trial val score: {best_trial['val_score']:.6e} "
    f"(RMSE_T={best_trial['val_rmse_T']:.6e}, RMSE_bc_V={best_trial['val_rmse_bc_V']:.6e})"
)
print(
    f"Test RMSE: T={math.sqrt(test_m_t['mse']):.6e}, bc_V={math.sqrt(test_m_v['mse']):.6e}; "
    f"R2: T={test_m_t['r2']:.6f}, bc_V={test_m_v['r2']:.6f}"
)

Section 5 split (OP19 excluded):
Train (10): ['OP15', 'OP14', 'OP01', 'OP13', 'OP10', 'OP03', 'OP02', 'OP16', 'OP09', 'OP12']
Val   (1): ['OP08']
Test  (1): ['OP11']


Full benchmark suite:  15%|█▍        | 10/67 [51:14<4:13:53, 267.26s/it, Section5 trial 9/24]

KeyboardInterrupt: 